# Exploring Financial Data using the Nasdaq Data Link API

In this project, we use the [Nasdaq Data Link API](https://docs.data.nasdaq.com/) to retrieve financial data from the **MER/F1** table (Mergent Global Fundamentals sample). We have two goals:

1. Analyze the trend of the **Accrued Expenses Turnover** ratio over time for different companies.
2. Examine how companies in different countries compare on this metric.

The API key is stored in a separate `config.py` file (`API_KEY = '...'`) so that it never appears in the notebook.

## Querying the API

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import config
api_key = config.API_KEY

api_url = 'https://data.nasdaq.com/api/v3/datatables/MER/F1.json'
parameters = {
    'api_key': api_key,
    'qopts.per_page': 10
}

json_data = requests.get(api_url, params=parameters).json()
print(json_data)

The response is a dictionary with two top-level keys:

- `datatable`, which holds `data` (a list of rows, each row being a list of values) and `columns` (a list of dictionaries with each column's `name` and `type`);
- `meta`, which holds `next_cursor_id` — used for pagination when a query returns more rows than one page can hold.

Each row describes one value of one financial indicator, reported by one company for one reporting period (e.g. Deutsche Bank AG, `2011-06-30`, `Q2`). To work with this data comfortably, we'll convert it to a DataFrame using the `data` and `columns` parts of the response.

## Processing the JSON Data into a DataFrame

In [ ]:
parameters['qopts.per_page'] = 10000
json_data = requests.get(api_url, params=parameters).json()

data = json_data['datatable']['data']
columns = [col['name'] for col in json_data['datatable']['columns']]

df_metric = pd.DataFrame(data, columns=columns)
print(df_metric.shape)
df_metric.head()

## Understanding the Dataset

In [ ]:
df_metric.info()

In [ ]:
necessary_columns = ['reportid', 'reportdate', 'reporttype', 'amount', 'longname', 'country', 'indicator', 'statement']

Why these columns:

- `longname` identifies the company, so we can compare companies with each other.
- `reportdate` (and `reporttype`) place each value in time, which we need for the trend analysis.
- `amount` holds the value of the financial metric itself.
- `indicator` tells us *which* metric the amount refers to, so we can isolate Accrued Expenses Turnover.
- `country` lets us compare companies by geographical region.
- `reportid` and `statement` help trace each value back to its report and financial statement.

Other columns, such as internal codes and address details, aren't needed for these two goals.

## Filtering the DataFrame for Analysis

In [ ]:
filtered_df = df_metric[necessary_columns]
filtered_df = filtered_df[filtered_df['indicator'] == 'Accrued Expenses Turnover'].copy()
print(filtered_df.shape)
filtered_df.describe()

There are **139 Accrued Expenses Turnover** values in our sample. The ratio varies a lot between reports:

- the **mean is 26.91**, but the **median is only 13.75** — a few very large values pull the mean up;
- values range from **4.69 to 229.81**, and the standard deviation (33.10) is larger than the mean;
- 75% of the values are below 31.78.

This right-skewed distribution suggests that most companies pay off their accrued expenses roughly 5–30 times per year, while a few companies have unusually high turnover in some periods. The variation likely reflects differences in industry, company size and how each company manages its short-term liabilities.

## Enhancing the DataFrame

In [ ]:
country_mapping = {
    'USA': 'United States of America',
    'DEU': 'Germany',
    'JPN': 'Japan',
    'CYM': 'Cayman Islands',
    'BHS': 'Bahamas',
    'IRL': 'Ireland',
    'GBR': 'United Kingdom'
}

def full_country_name(code):
    return country_mapping.get(code, code)

filtered_df['country_name'] = filtered_df['country'].apply(full_country_name)

updated_df = filtered_df.rename(columns={
    'reportid': 'report_id',
    'reportdate': 'report_date',
    'reporttype': 'report_type',
    'longname': 'company_name'
})
updated_df['report_date'] = pd.to_datetime(updated_df['report_date'])

updated_df['country_name'].value_counts()

The sample contains companies from seven countries. The United States has the most reports (31), followed by Japan and the Cayman Islands (27 each), Ireland (25), the Bahamas (19), Germany (6) and the United Kingdom (4).

Note that every country is represented by only one or two companies, so country-level comparisons below are really comparisons between a handful of companies.

## Understanding Financial Trends Over Time

In [ ]:
relevant_data = updated_df[['company_name', 'report_date', 'amount']].sort_values('report_date')

plt.figure(figsize=(12, 7))
for company in relevant_data['company_name'].unique():
    company_data = relevant_data[relevant_data['company_name'] == company]
    plt.plot(company_data['report_date'], company_data['amount'], marker='o', label=company)

plt.title('Accrued Expenses Turnover by Company Over Time')
plt.xlabel('Report Date')
plt.ylabel('Accrued Expenses Turnover')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize='small')
plt.tight_layout()
plt.show()

In [ ]:
summary = relevant_data.groupby('company_name')['amount'].agg(['count', 'min', 'max', 'mean']).round(2)
summary['first_report'] = relevant_data.groupby('company_name')['report_date'].min().dt.date
summary['last_report'] = relevant_data.groupby('company_name')['report_date'].max().dt.date
summary.sort_values('mean', ascending=False)

The trend plot and the summary table show very different behaviours across the eight companies (2010–2015):

- **Accenture plc** (6.47–8.69), **Deutsche Bank AG** (8.11–10.79) and **Canon Inc** (9.13–12.12) have low and very stable ratios.
- **Herbalife** (12.45–18.54) and **Apple Inc** (11.78–40.92, mean 27.78) sit in the middle; Apple's ratio fluctuates more from quarter to quarter.
- **Nielsen Holdings** has the highest average (113.52) but only six reports, ranging from 63.68 to 147.95.
- **Ultrapetrol (Bahamas)** is the most volatile company: its ratio goes from 4.69 up to 229.81, the widest range in the sample.

Stable, low ratios suggest consistent payment practices, while large swings (Ultrapetrol, Nielsen) indicate changes in how quickly accrued expenses are paid off, or changes in the underlying business.

## Geographical Region Analysis

In [ ]:
country_avg = updated_df.groupby('country_name')['amount'].mean().sort_values(ascending=False)
print(country_avg)

plt.figure(figsize=(10, 6))
country_avg.plot(kind='bar')
plt.title('Average Accrued Expenses Turnover by Country')
plt.xlabel('Country')
plt.ylabel('Average Accrued Expenses Turnover')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

By average Accrued Expenses Turnover, the **Bahamas (66.52)** comes first, followed by the **United States (44.38)** and the **United Kingdom (44.17)**. The Cayman Islands (15.06), Japan (10.38), Germany (9.29) and Ireland (7.26) have much lower averages.

These results should be interpreted with care. Each country average is driven by one or two companies — the Bahamas value is entirely Ultrapetrol, whose volatile ratio inflates the mean, and the US value combines Nielsen's very high ratio with Apple's moderate one. The chart tells us more about individual companies than about countries.

## Conclusion

- Using the Nasdaq Data Link API, we retrieved 10,000 rows from the MER/F1 table and isolated 139 **Accrued Expenses Turnover** values for 8 companies in 7 countries.
- The ratio is right-skewed: a median of 13.75 but a mean of 26.91, with values from 4.69 to 229.81.
- **Accenture, Deutsche Bank and Canon** have low, stable ratios; **Ultrapetrol and Nielsen** have high and volatile ones.
- The **Bahamas, the US and the UK** have the highest country averages, but each country is represented by only one or two companies, so the comparison reflects individual companies rather than regions.

Next steps: request more pages of data (using `next_cursor_id`) to include more companies per country, compare other indicators from the same table, and look at the trend per company relative to its own industry.